In [ ]:
import math
import cv2
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F

from module.classification_package.src.datamodule import ImageEmbeddingDataModule

class SafePoolingExtractor:
    def __init__(self):
        self.attn_maps = None

    def __call__(self, module, input, output):
        # Extract attn_out, which is returned second from our pooling layer
        if isinstance(output, tuple) and len(output) >= 2 and output[1] is not None:
            self.attn_maps = output[1].detach().cpu()

def visualize_all_attention_heads(input_tensor, model, patch_size=14):
    """
    Universal visualization function for ANY resolution (square or rectangle)
    """
    extractor = SafePoolingExtractor()
    target_layer = None

    tensor = input_tensor
    mean = torch.tensor([0.485, 0.456, 0.406])[:, None, None]
    std = torch.tensor([0.229, 0.224, 0.225])[:, None, None]
    img_np_rgb = tensor.cpu() * std + mean
    img_np_rgb = img_np_rgb.clamp(0, 1).permute(1, 2, 0).numpy()
    
    # Dimensions of the original input tensor
    orig_h, orig_w = img_np_rgb.shape[:2]

    for name, module in model.named_modules():
        if module.__class__.__name__ == 'ViTAttentionPooling':
            target_layer = module
            break
            
    if target_layer is None:
        print("❌ ViTAttentionPooling layer not found in the model!")
        return
        
    handle = target_layer.register_forward_hook(extractor)
    
    model.eval()
    device = next(model.parameters()).device
    
    batch_tensor = input_tensor
    if not isinstance(batch_tensor, torch.Tensor):
        batch_tensor = torch.from_numpy(np.asarray(batch_tensor))
    if batch_tensor.dim() == 3:
        batch_tensor = batch_tensor.unsqueeze(0)
    
    seg_logits = None
    
    try:
        with torch.no_grad():
            try:
                out = model(batch_tensor.to(device), return_attention_map=True)
            except TypeError:
                out = model(batch_tensor.to(device))
                
            if isinstance(out, tuple) and len(out) >= 4:
                seg_logits = out[3]
    finally:
        handle.remove() 
    
    attn_maps = extractor.attn_maps
    if attn_maps is None:
        print("❌ Hook failed to intercept the tensor. Check the pooling forward pass.")
        return

    # --- DIMENSION CORRECTION ---
    if attn_maps.ndim == 4:
        attn_maps = attn_maps[0].squeeze(-1) # -> [num_heads, N]
    elif attn_maps.ndim == 3:
        attn_maps = attn_maps[0].permute(1, 0) # [heads, N]
    elif attn_maps.ndim == 2:
        attn_maps = attn_maps.permute(1, 0) # [heads, N]
        
    num_heads = attn_maps.shape[0]
    total_tokens = attn_maps.shape[1]
    
    # =========================================================================
    # RECTANGLE SOLUTION: Calculate the grid directly from pixel dimensions
    # =========================================================================
    grid_h = orig_h // patch_size
    grid_w = orig_w // patch_size
    num_patch_tokens = grid_h * grid_w
    num_extra_tokens = total_tokens - num_patch_tokens
    
    if num_extra_tokens < 0:
        print(f"❌ Calculation error! Expected {num_patch_tokens} patches ({grid_h}x{grid_w}), but tensor only has {total_tokens} tokens.")
        return
    # =========================================================================

    has_seg = seg_logits is not None
    axes_needed = num_heads + 2 + (1 if has_seg else 0)
    
    # Create canvas
    fig, axes = plt.subplots(1, axes_needed, figsize=(4.0 * axes_needed, 4.0))
    if isinstance(axes, np.ndarray):
        axes = axes.flatten().tolist()
    elif not isinstance(axes, list):
        axes = [axes]
    
    image = img_np_rgb
    
    # 1. Plot original image
    axes[0].imshow(image)
    axes[0].set_title(f"Original ({orig_h}x{orig_w})", fontsize=12)
    axes[0].axis('off')
    
    normalized_maps = []
    
    # 2. Plot each attention head
    for h in range(num_heads):
        # Discard auxiliary tokens (cls, registers), keep only spatial patches
        patch_attention = attn_maps[h, num_extra_tokens:]
        
        # Reshape into the correct rectangular grid!
        map_2d = patch_attention.reshape(grid_h, grid_w).numpy()
        map_2d = map_2d.astype(np.float32)
        
        # Gamma correction for better visual contrast
        map_2d = np.power(map_2d, 0.4) 
        
        map_min, map_max = map_2d.min(), map_2d.max()
        if map_max - map_min > 1e-8:
            map_2d = (map_2d - map_min) / (map_max - map_min)
        else:
            map_2d = np.zeros_like(map_2d)
            
        normalized_maps.append(map_2d)
        
        # Resize attention map to the pixel dimensions of the image
        map_resized = cv2.resize(map_2d, (orig_w, orig_h), interpolation=cv2.INTER_CUBIC)
        
        axes[h+1].imshow(image)
        axes[h+1].imshow(map_resized, cmap='jet', alpha=0.55)
        axes[h+1].set_title(f"Head {h+1}", fontsize=12)
        axes[h+1].axis('off')
        
    # 3. Plot Union Attention (Total Coverage)
    stacked_maps = np.stack(normalized_maps, axis=0)
    comb_map_2d = np.max(stacked_maps, axis=0)
    comb_map_resized = cv2.resize(comb_map_2d, (orig_w, orig_h), interpolation=cv2.INTER_CUBIC)
    
    axes[num_heads + 1].imshow(image)
    axes[num_heads + 1].imshow(comb_map_resized, cmap='jet', alpha=0.55)
    axes[num_heads + 1].set_title("All Heads (Coverage)", fontsize=12, fontweight='bold', color='darkred')
    axes[num_heads + 1].axis('off')

    # 4. Plot Binary Segmentation Mask
    if has_seg:
        # Resize raw logits to the original image size
        seg_logits_resized = F.interpolate(
            seg_logits.cpu(), size=(orig_h, orig_w), mode='bilinear', align_corners=False
        )
        
        seg_prob = torch.sigmoid(seg_logits_resized[0, 0]).numpy()
        bin_mask = (seg_prob > 0.5).astype(np.uint8) 
        
        axes[-1].imshow(image)
        # Overlay the binary mask
        axes[-1].imshow(bin_mask, cmap='jet', alpha=0.55, vmin=0, vmax=1)
        axes[-1].set_title("Pred Mask", fontsize=12, fontweight='bold', color='darkgreen')
        axes[-1].axis('off')
        
    plt.tight_layout()
    plt.show()

In [ ]:

db_mdule = ImageEmbeddingDataModule(
        dataset_name = '',
        labels_path = '',
        class_mapping_path= '',
        val_batch_size = 64,
        classes_per_batch = 32,
        samples_per_class = 4,
        image_size = (154, 420),
        train_tags = ['val'] + [f"train_{i}" for i in range(100000)],
        augmentation_preset = 'base',
        val_tags = ['val'],
        instance_data = True,
        bg_removal_prob = 0.0,
        resize_strategy = 'pad',
        alignment_method = 'horizontal',
        cache_dir = '',
        use_cache = True
    )
db_mdule.setup()

In [ ]:
from module.classification_package.src.model_v2 import StableEmbeddingModelViT

MODEL = ""

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_a_backbone = StableEmbeddingModelViT.load_from_checkpoint(MODEL)



In [ ]:
ds_loader = db_mdule.train_dataloader()

# id_to_visualize = [20865]
for images, targets, _, batch_meta in ds_loader:
    for input_image in images:
        print(input_image.shape)
        visualize_all_attention_heads(
            input_tensor=input_image.cpu(),
            model=model_a_backbone
        )
        break
    break